# Computational Text Analysis Part II: 
## This time it's personal

In this lecture, I'm going to introduce a few more advanced computational text analysis approaches: Topic modeling, word embeddings, and LLM-based classifications.

## Topic Modeling

The first example is topic modeling. The idea of topic modeling is to group documents (in this case, posts) which are similar to each other, and to characterize those groups somehow. In that sense, it is similar to qualitative inductive coding.

There are a number of approaches. I'm going to show you a very vanilla version of BERTopic, which is a new, fancy approach which uses a large language model (LLM) in order to understand the semantic meaning of sentences in a corpus.

To install it, run `conda install bertopic` in the terminal.

The BERTopic library is really great, and has [great documentation and a website here](https://maartengr.github.io/BERTopic/index.html).

**Important Note on Long Documents:** By default, BERTopic uses sentence transformers that have a 512 token limit (~300-400 words). Longer texts get truncated, meaning you only analyze the beginning. For longer documents like Reddit posts, you should either:
1. Use a longer-context embedding model (see below - recommended)
2. Intelligently truncate or chunk your documents
3. Focus only on titles or shorter sections

The first step is to load a model. (Note that `hdbscan_model = ...` line is optional, and sets some parameters which help to avoid having lots of topics. The `representation_model` is also optional, but can help to identify more representative terms for each topic. The `embedding_model` parameter lets you use models that handle longer text.)

In [4]:
# BERTopic example
from hdbscan import HDBSCAN
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

# Use a model with longer context (8192 tokens instead of 512)
# This model is specifically designed for longer documents
embedding_model = SentenceTransformer("jinaai/jina-embeddings-v2-small-en", trust_remote_code=True)

# Load a pre-trained BERT model
hdbscan_model = HDBSCAN(min_cluster_size=25, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
representation_model = KeyBERTInspired()
topic_model = BERTopic(language="english", verbose=True, 
                       hdbscan_model=hdbscan_model, 
                       representation_model=representation_model,
                       embedding_model=embedding_model)  # Add the embedding model

Let's load the subreddit data from last week. Because there are more `r/politics` posts than the other subreddits, we'll focus on those.

This next piece of code loads the data and trains the model. For speed, we'll pre-compute the embeddings separately - this allows us to reuse them if we want to adjust other parameters later. The whole process may take a few minutes to run, but BERTopic has a nice progress bar so you know that it's working.

We're also going to just look at a subset of the data for this example, to speed things up. You can always run it on the whole dataset if you have the time. 

In [5]:
sr = pd.read_csv('https://raw.githubusercontent.com/jdfoote/Intro-to-Programming-and-Data-Science/refs/heads/master/resources/data/sr_post_data.csv')

sr = sr[sr.subreddit == 'politics']
# First we change NAs and removed/deleted to empty strings
sr.loc[(pd.isna(sr.selftext)) | (sr.selftext.isin(['[removed]', '[deleted]'])), 'selftext'] = ''
sr['all_text'] = sr.title + ' ' + sr.selftext


dataset = sr.all_text.to_list()

In [20]:
# Pre-compute embeddings for faster processing and reusability
embeddings = embedding_model.encode(dataset, show_progress_bar=True)

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

In [21]:
topics, probs = topic_model.fit_transform(dataset, embeddings=embeddings)

2026-02-11 12:04:43,200 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-11 12:04:50,185 - BERTopic - Dimensionality - Completed ✓
2026-02-11 12:04:50,186 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-11 12:04:50,249 - BERTopic - Cluster - Completed ✓
2026-02-11 12:04:50,252 - BERTopic - Representation - Extracting topics from clusters using representation models.
2026-02-11 12:05:08,998 - BERTopic - Representation - Completed ✓


BERTopic decides how many topics are appropriate, and assigns each document to a topic. It also includes a "miscellaneous" topic (`Topic -1`) for documents that don't fit very well. This can include quite a few documents, as below.

In [22]:
topic_model.get_topic_freq()

,Topic,Count
5,-1,611
10,0,330
0,1,282
2,2,205
7,3,129
9,4,129
1,5,114
8,6,85
16,7,73
13,8,66


There are a bunch of cool visualizations and tools for understanding the topics. There are a bunch of them shown [on the BERTopic website](https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_topics.html). Here are a few.

This first one visualizes the topics. We can see that they are fairly clustered.

In [23]:
topic_model.visualize_topics()

This shows the top words and their probabilities for each of the top `n` topics

In [24]:
topic_model.visualize_barchart(top_n_topics=15)

We can also visualize how much topics are used over time.

In [25]:
timestamps = sr[sr.subreddit=='politics'].date.to_list()
topics_over_time = topic_model.topics_over_time(dataset, timestamps, nr_bins=30, global_tuning=True)

30it [03:36,  7.23s/it]


In [26]:
topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=5)

### Qualitative Analyses

I think it's really vital to get back into the actual text data in order to make sure that the topics really represent what you think they do. One way to do that is to extract the documents most closely associated with each topic.

First, we get info about each document and how well it matches the topic.

In [27]:
doc_df = topic_model.get_document_info(dataset)

# Remove the -1 topic, which is the "garbage" topic
doc_df = doc_df.loc[doc_df.Topic != -1]

Then, we sort by "Probability", which is the likelihood that the document belongs to the assigned topic, group by topic, and take the top `num_docs` documents for each topic.

I print these, but in practice what you probably want to do is save the output, so you can look at it in Excel or similar. (i.e., `top_docs.to_csv('top_documents.csv')`)

In [28]:
num_docs = 20
top_docs = doc_df.sort_values('Probability', ascending=False).groupby('Topic').head(num_docs)
top_docs = top_docs.sort_values('Topic').loc[:, ['Topic', 'Probability', 'Document']]

In [29]:
for topic, group in top_docs.groupby('Topic'):
    print(f"Topic {topic}")
    print(group.Document.values)
    print("\n\n")

Topic 0
["Hannity questions whether FAA grounded drones at southern border to 'cover up for Biden's failures' "
 'Florida Democrat says vaccines, masks are key to small-business recovery '
 'Biden order would allow government to require quarantine for measles cases '
 'Biden’s Vaccine Mandate Is Unconstitutional. The media were quick to criticize Trump when he claimed similar powers last year. '
 'Will the Biden Administration Mandate Vaccines for Flying? That sound you hear is the gang at Fox News screaming in a pitch only dogs can hear. '
 'Oklahoma Pastor says he has signed thousands of religious exemption forms for the COVID-19 vaccine '
 'Hospital staff must swear off Tylenol, Tums to get religious vaccine exemption '
 "Ron DeSantis' 'Disastrous' COVID-19 Response Ripped In Viral 'Florida Is Vietnam' Video | The Florida Republican's response to the pandemic is hammered in author Don Winslow's latest video, which has topped 1 million views. "
 'A group of experts pens paper disagre

### EXERCISE 1

Where topic modeling really shines is in analyzing longer texts - for example, the subreddit [changemyview](https://www.reddit.com/r/changemyview/) has fairly long posts where people explain a controversial view that they hold.

Try to figure out how to get a few hundred posts from changemyview using PRAW, and run a topic model on them, where the selftext of each post is a document.

## Word Embeddings

The next method is word embeddings. Word embeddings crete a multidimensional "space" and then place words in that space based on the words that they appear near in a corpus. There are a bunch of complex versions of word embeddings, and complex uses for them. Indeed, BERTopic uses word embeddings, as do LLMs. 

The embeddings themselves can also be interesting, as we can think of them as putting words into a contextualized semantic space. We can then compare how different groups or communities contextualized different terms or concepts differently.

I'm going to teach you a simple version of word embeddings called Word2Vec. In this example, we'll build the model from scratch, but another option is to use something like BERT to build on a pre-trained model.

Much of what follows is borrowed from [Laura Nelson's wonderful example](https://github.com/lknelson/DH-Institute-2017/blob/d20246758d6da88dfedbad2e75933ad4ef370930/07-Word2Vec/Word2Vec.ipynb).

We will use Laura's code as template to look at differences between some recent comments on `r/Purdue` and `r/IndianaUniversity`

In [9]:
import numpy as np
#import pandas as pd
#from sklearn.metrics import pairwise
#from sklearn.manifold import MDS, TSNE

import gensim
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

from string import punctuation


In [10]:

def fast_tokenize(text):
    
    # Get a list of punctuation marks
    
    lower_case = text.lower()
    
    # Iterate through text removing punctuation characters
    no_punct = "".join([char for char in lower_case if char not in punctuation])
    
    # Split text over whitespace into list of words
    tokens = no_punct.split()
    
    return tokens


In [11]:
def tokenize_sr(df, sr_name):
    sr_data = df[df.subreddit==sr_name].body.to_list()
    sr_data = [fast_tokenize(text) for text in sr_data]
    sr_data = [text for text in sr_data if len(text) > 0]
    return sr_data

df = pd.read_csv('https://raw.githubusercontent.com/jdfoote/Intro-to-Programming-and-Data-Science/refs/heads/master/resources/data/purdue_iu_comments.csv')
purdue_data = tokenize_sr(df, 'Purdue')
iu_data = tokenize_sr(df, 'IndianaUniversity')


Word2Vec actually has two different options for algorithms. CBOW (Continuous Bag of Words) and Skip-Gram.

I won't focus on the details here. In general, CBOW is is faster and does well with frequent words, while Skip-Gram can be better for rare words.

Parameters for the `gensim` `Word2Vec` function that you might want to adjust:

* vector_size: Number of dimensions for embedding model
* window: Number of context words to observe in each direction
* min_count: Words must appear this many times to be included
* max_vocab_size: Maximum number of words to consider (will remove less frequent words)
* sg (Skip-Gram): '0' indicates CBOW model; '1' indicates Skip-Gram
* alpha: Learning rate
* epochs: Number of passes (iterations) through dataset

Note: The code below uses the default for all values except for `sg`. In general, you probably don't need to change any of the parameters.

In [33]:
purdue_model, iu_model = (gensim.models.Word2Vec(x, vector_size=100, window=5,
                               min_count=5, max_vocab_size=None, sg=1, alpha=0.025, epochs=5) for x in [purdue_data, iu_data])

We should now have vectors for each common word that appears in the data. Each word is represented by 100 numbers (its location in the 100-dimension meaning space)

In [34]:
purdue_model.wv['study']

array([ 0.0270892 ,  0.17593269,  0.13596722, -0.2707213 ,  0.11681859,
       -0.08742422,  0.24871293,  0.16814536, -0.07707875, -0.39426914,
        0.00646493, -0.13805738,  0.2898589 , -0.13528988,  0.21153615,
       -0.2805445 ,  0.11007001, -0.3163033 , -0.5060455 , -0.410239  ,
        0.32233012, -0.189152  , -0.18853615,  0.13757329, -0.15857735,
        0.12744933, -0.072671  , -0.27573258, -0.14169806, -0.07534803,
        0.1566296 , -0.04327328, -0.2617183 , -0.409631  , -0.31037417,
        0.38364857, -0.05929216, -0.33501592, -0.2065673 ,  0.10030438,
       -0.0730049 , -0.14905803, -0.02711712, -0.01961783, -0.11432902,
       -0.24909343,  0.25876677,  0.3183575 ,  0.4672539 ,  0.40803257,
        0.2867621 , -0.26415664, -0.05492304,  0.1372186 , -0.01989349,
       -0.32517728,  0.39422208, -0.03501939,  0.07575988,  0.50735754,
        0.15030982,  0.35604176,  0.105998  , -0.01669323, -0.31365293,
       -0.1075177 ,  0.02475536,  0.1890459 , -0.6944327 ,  0.27

We can now do things like look at which terms are most similar to a given topic in both communities. For example, this shows the words most similar to "sports" and "studying"

In [35]:
print(f"Purdue similar words to study: {purdue_model.wv.most_similar('studying')}")
print(f"IU similar words to study: {iu_model.wv.most_similar('studying')}")

Purdue similar words to study: [('electives', 0.9255090355873108), ('draw', 0.9237362742424011), ('dates', 0.9227925539016724), ('masters', 0.9224894046783447), ('quizzes', 0.9203352332115173), ('paragraph', 0.9161503314971924), ('hoops', 0.9131243228912354), ('laptop', 0.9124829173088074), ('monday', 0.9122766256332397), ('admittance', 0.911828875541687)]
IU similar words to study: [('feeling', 0.9399603605270386), ('age', 0.9134945273399353), ('guaranteed', 0.9073247909545898), ('moment', 0.9068223237991333), ('transferring', 0.9059239625930786), ('dropping', 0.9056227207183838), ('apart', 0.9051257371902466), ('hoping', 0.9051229953765869), ('extracurriculars', 0.9048848748207092), ('teacher', 0.9048764109611511)]


In [36]:
print(f"Purdue similar words to sports: {purdue_model.wv.most_similar('sports')}")
print(f"IU similar words to sports: {iu_model.wv.most_similar('sports')}")

Purdue similar words to sports: [('sections', 0.9290592670440674), ('farms', 0.9247890710830688), ('exclusively', 0.924749493598938), ('articles', 0.9235065579414368), ('rides', 0.9230427145957947), ('tickets', 0.921772837638855), ('shifting', 0.9217640161514282), ('stealing', 0.9206180572509766), ('sunsets', 0.9200796484947205), ('heading', 0.9186500310897827)]
IU similar words to sports: [('exercise', 0.8638362884521484), ('rec', 0.8587436079978943), ('uptown', 0.8563314080238342), ('tailgating', 0.8494824171066284), ('dinner', 0.8414848446846008), ('lines', 0.8390969038009644), ('distance', 0.8378159403800964), ('lunch', 0.8376121520996094), ('breakfast', 0.8325849771499634), ('art', 0.8314971923828125)]


### Exercise

Identify topics where you think IU and Purdue commenters might differ and figure out how to display those differences.

## Using LLMs for research

The last thing I want to show you is some example code for using LLMs (like ChatGPT or Claude) in your work.

They are incredible, flexible tools, which have a broad semantic understanding of texts, and can be used in a lot of the same ways as a trained undergraduate.

For example, let's say we wanted to identify the different hobbies that people do at each school.

In a sense, what we're doing is programming an LLM agent using natural language. So, we want to come up with a prompt. I'll show you "few-shot" prompting, which gives a few examples for the agent. This can often be helpful, especially when a task might be ambiguous. Unlike most of the programs we've written so far, you may receive different results with even small changes to a prompt. It's a stochastic process.

We'll use Purdue's RCAC (Rensselaer Center for Advanced Computing) to access LLMs. RCAC provides free API access to various models for Purdue students and researchers.

To get started:
1. Visit [https://www.rcac.purdue.edu/](https://www.rcac.purdue.edu/) and follow their instructions for API access
2. Get your API key from the RCAC portal
3. Save it in a file called `rcac_credentials.py` with: `api_key = "your_key_here"`

The below code uses the OpenAI-compatible API that RCAC provides.

In [1]:
import requests
import rcac_credentials
import time
import json

# RCAC API endpoint
RCAC_API_URL = "https://genai.rcac.purdue.edu/api/chat/completions"
API_KEY = rcac_credentials.api_key

# Available models: "llama3.1:latest", "gpt-oss:120b", etc. Check RCAC docs for current options
MODEL = "llama4:latest"
MODEL = "gpt-oss:120b"

def get_classifications(comments, num_comments):
    """Call RCAC API to classify hobbies from comments.
    
    Args:
        comments: List of comment strings OR list of (index, comment) tuples
        num_comments: Number of comments being classified
    """
    # Handle both indexed tuples and plain strings
    if comments and isinstance(comments[0], tuple):
        # Already indexed: format with original indices
        comments_text = "\n".join([f"{idx}. {comment}" for idx, comment in comments])
    else:
        # Plain strings: format with 0-based indices
        comments_text = "\n".join([f"{i}. {comment}" for i, comment in enumerate(comments)])
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    
    body = {
        "model": MODEL,
        "messages": [
            {
                "role": "user",
                "content": f"""You are an AI assistant tasked with analyzing Reddit comments from Purdue University and Indiana University (IU) subreddits to identify hobbies and leisure activities.

I will provide you with {num_comments} comments. Your task is to process each one individually and return a structured JSON array.

### Input Comments (Each includes an ID and a comment):
<reddit_comments>
{comments_text}
</reddit_comments>

### Your Task:
For each comment, create a JSON object containing:
1. "id": The index number of the comment (as shown above).
2. "comment_preview": The first 5 words of the original comment (to ensure alignment).
3. "reasoning": A very brief note (5 words max) on why this is or isn't a hobby.
4. "hobby": The extracted hobby/activity, or an empty string "" if none is found.

### Extraction Rules:
- A hobby/activity is any pastime or interest (sports, clubs, gaming, etc.) other than academic studies.
- Be generous in your interpretation, but do not make assumptions (e.g., "going to class" is not a hobby; "playing intramural soccer" is).
- If multiple hobbies are mentioned, extract only the most prominent one.
- **CRITICAL**: You must return exactly {num_comments} objects in the array, in the same order as the input comments.

### Output Format:
Return ONLY a valid JSON array of objects. Do not include markdown backticks, explanations, or any text before or after the JSON.

Example:
[
  {{"id": 0, "comment_preview": "Basketball is the best sport...", "reasoning": "Mentions sports", "hobby": "watching basketball"}},
  {{"id": 1, "comment_preview": "Math 231 is super tough...", "reasoning": "Academic complaint", "hobby": ""}}
]

Begin your analysis.
"""
            }
        ],
        "stream": False
    }
    
    try:
        response = requests.post(RCAC_API_URL, headers=headers, json=body)
        if response.status_code == 200:
            result = response.json()
            classification_json = result['choices'][0]['message']['content']
            return classification_json
        else:
            raise Exception(f"Error: {response.status_code}, {response.text}")
    except Exception as e:
        print(f"Error {e}: Retrying after 30 seconds")
        time.sleep(30)
        return get_classifications(comments, num_comments)

In [12]:
get_classifications(['Test', 'I love playing basketball at Purdue', 'IU has great hiking trails'], 3)

'[\n  {"id":0,"comment_preview":"Test","reasoning":"No hobby mentioned","hobby":""},\n  {"id":1,"comment_preview":"I love playing basketball at","reasoning":"Sports activity mentioned","hobby":"basketball"},\n  {"id":2,"comment_preview":"IU has great hiking trails","reasoning":"Mentions outdoor activity","hobby":"hiking"}\n]'

The code below is one version of how you might do this, and is the result of running into some issues with other approaches.

I found out that if you have too many comments, then it doesn't always keep track of which is which, so I batched them into groups of 10.

Then, if it still returns the wrong number, I go one comment at a time.

Also, note that I write out comments directly to a file, and skip to where I left off. This is a good practice so you don't have to start over if there's a network error.

In [35]:
# Get 1000 comments from each subreddit
sample = df.groupby('subreddit').sample(100, random_state=2026)

# Make a list of (index, comment) tuples
comments = sample.body.to_list()
index = sample.index.to_list()
indexed_comments = list(zip(index, comments))

# Batch the comments into groups of 10
batch_size = 10
comment_batches = [indexed_comments[i:i+batch_size] for i in range(0, len(indexed_comments), batch_size)]

In [37]:
import csv
hobbies_fn = 'hobbies.csv'
try:
    with open(hobbies_fn, 'r') as f:
        hobbies = [line.strip() for line in f]
except FileNotFoundError:
    hobbies = []
hobbies_count = len(hobbies)

batch_num = 0
with open(hobbies_fn, 'a') as f:
    out_csv = csv.writer(f)
    for batch in comment_batches:
        batch_num += 1
        batch_start = (batch_num - 1) * batch_size
        batch_end = batch_start + len(batch)
        
        # Skip batches we've already processed
        if batch_end <= hobbies_count:
            continue
            
        # If we're partway through a batch, only process remaining comments
        if batch_start < hobbies_count:
            skip = hobbies_count - batch_start
            batch = batch[skip:]
            print(f"Resuming batch {batch_num} at comment {skip + 1}")

        print(f"Processing batch {batch_num} of {len(comment_batches)} ({len(batch)} comments)")
        
        # Pass batch with original indices to LLM
        response_text = get_classifications(batch, len(batch))
        print(response_text)
        print(batch)
        
        try:
            curr_hobbies = json.loads(response_text)
        except json.JSONDecodeError as e:
            print(f"JSON decode error: {e}")
            print(f"Response was: {response_text}")
            curr_hobbies = [{"id": original_idx, "hobby": ""} for original_idx, _ in batch]
        
        # Check if we got the right number of items and all are dicts
        if len(curr_hobbies) != len(batch) or not all(isinstance(item, dict) for item in curr_hobbies):
            print(f"Warning: Expected {len(batch)} dict objects but got {len(curr_hobbies)} items or wrong types. Processing individually...")
            curr_hobbies = []
            for original_idx, comment in batch:
                response_text = get_classifications([comment], 1)
                try:
                    hobby_list = json.loads(response_text)
                    # Extract hobby from the response (handle both dict and string responses)
                    if hobby_list and isinstance(hobby_list[0], dict):
                        hobby = hobby_list[0].get("hobby", "")
                    else:
                        hobby = ""
                except (json.JSONDecodeError, IndexError, TypeError):
                    print(f"Failed to parse response for comment {original_idx}: {response_text}")
                    hobby = ""
                curr_hobbies.append({"id": original_idx, "hobby": hobby})
        
        # Write results to CSV (sorted by id to maintain order)
        for hobby_obj in curr_hobbies:
            if isinstance(hobby_obj, dict):
                id = hobby_obj.get("id", "")
                comment_preview = hobby_obj.get("comment_preview", "")
                hobby = hobby_obj.get("hobby", "")
            else:
                hobby = ""
            out_csv.writerow([id, comment_preview, hobby])

Processing batch 1 of 20 (10 comments)
[
  {"id": 8267, "comment_preview": "Just apply test optional. If", "reasoning": "No hobby mentioned", "hobby": ""},
  {"id": 8597, "comment_preview": "No one got sniped (yet)!", "reasoning": "Implied gaming reference", "hobby": "gaming"},
  {"id": 20685, "comment_preview": "laundry in the dorms is", "reasoning": "No hobby mentioned", "hobby": ""},
  {"id": 2742, "comment_preview": "All GREAT suggestions!! ❤️", "reasoning": "No hobby mentioned", "hobby": ""},
  {"id": 6890, "comment_preview": "They will slowly choke off", "reasoning": "No hobby mentioned", "hobby": ""},
  {"id": 8488, "comment_preview": "Yeah, tailgating is mainly", "reasoning": "Mentions tailgating activity", "hobby": "tailgating"},
  {"id": 23989, "comment_preview": "Amr is flipping awesome. Not", "reasoning": "No hobby mentioned", "hobby": ""},
  {"id": 19913, "comment_preview": "Someone stating how they see", "reasoning": "No hobby mentioned", "hobby": ""},
  {"id": 20608, "co

We can then put the hobbies back into the original dataframe, and do things like filter by them, compare them across campuses, etc.

In [38]:
sample

,subreddit,body,author,score,created_utc,post_id
8267,IndianaUniversity,Just apply test optional. If he has a good GPA...,andrew-js99,1,1.725418e+09,1f8b9ym
8597,IndianaUniversity,No one got sniped (yet)!,RespectfullyNoirs,2,1.724848e+09,1f1s056
20685,IndianaUniversity,laundry in the dorms is the easiest to get to ...,Foursporks,6,1.720386e+09,1dxjwjm
2742,IndianaUniversity,All GREAT suggestions!! ❤️,No-Ranger-3299,2,1.727458e+09,1fqbpu8
6890,IndianaUniversity,They will slowly choke off everything in the u...,BloomingtonResists,19,1.727914e+09,1fuj1j4
...,...,...,...,...,...,...
3909,Purdue,Nobody is focusing on that trust me,Mangatomboy,5,1.729040e+09,1g4fgfq
5185,Purdue,Not like Card was making a bit of difference.,Cubs2015WS,1,1.728735e+09,1g1hl7b
15231,Purdue,Where/how to apply to be RA though?,HazaarKhwaisheinAisi,1,1.727693e+09,1fn03hg
4998,Purdue,I will be a sophomore next year and this syste...,AttisTheFarmer1,24,1.728761e+09,1g25y5x


In [39]:
hobbies_df = pd.read_csv(hobbies_fn, names=["id", "comment_preview", "hobby"])
sample_with_hobbies = sample.merge(hobbies_df, left_on=sample.index, right_on="id")

In [40]:
sample_with_hobbies.to_csv('purdue_iu_comments_hobbies.csv', index=False)

In [45]:
sample_with_hobbies[sample_with_hobbies.hobby != ''].groupby('subreddit').hobby.count()

subreddit
IndianaUniversity    13
Purdue               14
Name: hobby, dtype: int64